# 🗓 Calendar Agent — Kaggle LoRA Training

Colab 대안. Kaggle은 별도 GPU quota를 제공 (주 30h, T4×2 또는 P100 16GB).

## 실행 전 준비 (한 번만)

### 1. Kaggle 설정 (우측 패널)
- **Accelerator**: `GPU T4 x2` 또는 `GPU P100` 선택
- **Internet**: `On` (외부 패키지·HF 모델 다운로드용)
- **Persistence**: `Variables and Files` (선택 — 세션 재시작 시 일부 보존)

### 2. 데이터 업로드 (Kaggle Dataset으로)
1. https://www.kaggle.com/datasets → `New Dataset`
2. 3개 파일 업로드:
   - `train.jsonl` (로컬 `D:\calendar-agent\data\processed\train.jsonl`)
   - `val.jsonl` (동)
   - `golden.jsonl` (로컬 `D:\calendar-agent\data\eval\golden.jsonl`)
3. Dataset 이름: `calendar-messages` (또는 임의)
4. Visibility: **Private**
5. Create

### 3. 이 노트북에 Dataset 첨부
- 우측 패널 `+ Add Input` → `Datasets` 탭 → 위에서 만든 dataset 검색 → 첨부
- 첨부되면 `/kaggle/input/datasets/sooryongbyun/calendar-messages/` 경로에 파일 보임

### 4. GitHub PAT 발급 (Private repo clone용)
https://github.com/settings/tokens/new?type=classic — repo scope, 7 days

---

준비 끝나면 셀을 위에서 아래로 순서대로 실행. 예상 시간: ~1시간.

## 0. GPU 확인

`Tesla T4` 또는 `Tesla P100` 표시되어야 함. 안 보이면 우측 패널 Accelerator 다시.

In [ ]:
!nvidia-smi

## 1. 레포 clone (Private)

Kaggle은 working dir이 `/kaggle/working/`. 그 안에 clone.

In [ ]:
import os, getpass

%cd /kaggle/working
if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/sooryong/calendar-agent.git
    token = None
else:
    print('calendar-agent already cloned, force-syncing to origin/main…')
    !cd calendar-agent && git fetch origin && git reset --hard origin/main

%cd /kaggle/working/calendar-agent
!git log --oneline -1

## 2. 학습 의존성 설치 + Colab/Kaggle 호환 cleanup

Colab과 동일 — install + torchao 제거.

In [ ]:
# 설치. ⚠ Kaggle 사전설치 RAPIDS(cudf/cuml/dask-cuda)가 numba/cuda-core를 옛 버전에 고정해 둬서
#   "X requires ... which is incompatible" 충돌 경고가 뜨지만, 우리 학습은 RAPIDS 미사용 → 무해.
#   아래 grep으로 그 충돌 보고 줄만 숨김(실제 설치 실패 메시지는 패턴이 달라 그대로 보임).
!pip install -q -e .[train] 2>&1 | grep -vE "which is incompatible|pip's dependency resolver does not|source of the following dependency"
!pip uninstall torchao -y -q   # PEFT가 거부하는 구버전 torchao 제거 (LoRA엔 불필요)
import os, torch
os.environ["WANDB_DISABLED"] = "true"
# CUDA_VISIBLE_DEVICES 설정 안 함 — DDP(torchrun)가 2 GPU를 다 써야 함.
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "ngpu:", torch.cuda.device_count())

## 3. 데이터 확인 (repo에 포함 — clone으로 따라옴, Kaggle 데이터셋 불필요)

In [ ]:
# 데이터는 repo(git)에 포함 → clone으로 따라옴 (현 라운드 train + golden 50). 데이터셋 첨부 불필요.
for p in ["data/processed/train.jsonl", "data/processed/val.jsonl", "data/eval/golden.jsonl"]:
    n = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"{p}: {n} records")

## 5. 학습 실행

먼저 **라운드 확인 셀**을 실행하면 현재 라운드·데이터 건수가 학습 없이 바로 표시된다. 확인 후 **학습 실행 셀**을 돌린다. (라운드는 `CONFIG` 한 줄로 변경)

T4×2 DDP(torchrun)로 ~1시간. Kaggle 세션 한도 9시간 — 무관.

In [ ]:
# ── 라운드 확인 (이 셀만 실행하면 학습 없이 현재 라운드가 바로 표시됨) ──
#   라운드는 config로 결정: 0.5B(fallback) configs/train.yaml / Qwen3-0.6B configs/train_qwen3_0_6b.yaml
CONFIG = 'configs/train_qwen3_0_6b.yaml'

import yaml
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
print(f"★ 현재 라운드 : {_cfg['run_name']}")
print(f"  베이스 모델 : {_mcfg['hf_id']}")
print(f"  output_dir : {_cfg['output_dir']}")
print(f"  유효 배치   : {_cfg['per_device_train_batch_size'] * _cfg['gradient_accumulation_steps']} (T4×2 DDP는 accum 자동분할로 유지)")
_n = sum(1 for l in open(_cfg['train_data'], encoding='utf-8') if l.strip())
print(f"  {_cfg['train_data']}: {_n}")

## 5.1 사전점검 — 학습/추론 프롬프트 정렬 확인 (학습 전 권장)

Qwen3는 thinking 모델이라 chat-template이 assistant 턴에 빈 `<think></think>`를 넣는다(**정상·의도된 설계**). 핵심은 *제거*가 아니라 **학습 렌더 == 추론 프롬프트 + gold JSON** 정렬이다 — `eval_model.infer`가 `enable_thinking=False`로 같은 빈 think를 프리필하므로 맞아야 한다.
'라운드 확인' 셀을 먼저 실행한 뒤 이 셀을 돌려라. `OK` 면 학습 진행, `assert` 멈추면 분포 불일치(템플릿 점검).


In [ ]:
# ── 사전점검: 학습 렌더 == 추론 프롬프트 + gold JSON (Qwen3 non-thinking 정렬) ──
#   '라운드 확인' 셀을 먼저 실행해 _mcfg를 채운 뒤 이 셀 실행.
from transformers import AutoTokenizer
import json
_tok = AutoTokenizer.from_pretrained(_mcfg['hf_id'], trust_remote_code=_mcfg.get('trust_remote_code', False))
_user = "<채널: KakaoTalk>\n<메시지>\n내일 3시 회의\n</메시지>"
_gold = json.dumps({'has_schedule': True, 'events': []}, ensure_ascii=False)
_sys  = _mcfg['system_prompt']
_full = [{'role':'system','content':_sys}, {'role':'user','content':_user}, {'role':'assistant','content':_gold}]
_pre  = [{'role':'system','content':_sys}, {'role':'user','content':_user}]
# 학습 렌더 (train_lora.to_chat와 동일)
_train = _tok.apply_chat_template(_full, tokenize=False, add_generation_prompt=False)
# 추론 렌더 (eval_model.infer와 동일: thinking 모델이면 enable_thinking=False)
_extra = {}
if 'enable_thinking' in (getattr(_tok, 'chat_template', None) or ''):
    _extra['enable_thinking'] = False
_infer = _tok.apply_chat_template(_pre, tokenize=False, add_generation_prompt=True, **_extra)
print('--- 학습 렌더 ---'); print(_train)
print('--- 추론 프롬프트 ---'); print(_infer)
_aligned = _train.startswith(_infer)
_tail = _train[len(_infer):] if _aligned else ''
_json_next = _tail.lstrip().startswith('{')
print()
print('정렬:', _aligned, '| 추론 직후 JSON:', _json_next, '->', 'OK: 학습 진행' if (_aligned and _json_next) else 'FAIL: 분포 불일치')
assert _aligned and _json_next, '학습 렌더가 (추론 프롬프트 + gold JSON)와 불일치 — train/eval 프롬프트 정렬 깨짐'


In [ ]:
# ── 학습 실행 (위 '라운드 확인' 셀 먼저 실행해 CONFIG 설정) ──
# DDP: T4 2개를 진짜로 병렬 사용 (torchrun, DataParallel 아님)
print(f"학습 시작 → {CONFIG} (라운드: {_cfg['run_name']})")
!torchrun --nproc_per_node=2 scripts/train_lora.py --config {CONFIG}

## 5.5 학습 직후 평가 (Kaggle GPU, 선택)

merge 후 골든셋으로 바로 평가해 `time_match_rate`/`final_score`를 확인한다. 로컬 CPU 평가(약 14분)보다 GPU가 훨씬 빠르다. 결과 json은 아래 6번 zip에 포함된다.

In [ ]:
# CONFIG에서 모든 경로 자동 인식 — base는 model_config의 hf_id에서 읽음.
import yaml, os
_cfg = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
BASE = _mcfg['hf_id']                          # model_config의 hf_id 자동
LORA_DIR = _cfg['output_dir']                  # 예: models/lora/r12-qwen0.5b
NAME = os.path.basename(LORA_DIR)              # r12-qwen0.5b
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON = f'logs/eval_{NAME}.json'
GOLDEN = _cfg.get('eval_golden', 'data/eval/golden.jsonl')
ZIP = f'/kaggle/working/lora_{NAME}.zip'       # 모델별로 구분
print(f'base={BASE}\nlora={LORA_DIR}\ngolden={GOLDEN}\nzip={ZIP}')

# merge → 실 골든셋 평가 (Kaggle GPU)
!python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON}
print(f'--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 6. 결과 패키징 (Kaggle Output 자동 노출)

Kaggle은 `/kaggle/working/` 안의 파일을 노트북 우측 패널 `Output`에서 다운로드 가능.
zip을 `/kaggle/working/` 루트에 두면 됨.

In [ ]:
!ls -la {LORA_DIR}/
!zip -r {ZIP} {LORA_DIR} {CONFIG} configs/lora.yaml {_cfg['model_config']} {EVAL_JSON}
!ls -la {ZIP}

## 다운로드 방법

**FileLink는 Kaggle 프록시에서 동작하지 않는다(404 남).** 아래 방법을 쓸 것:

1. **우상단 `Save Version` → `Quick Save`** (재실행 안 함, 현재 `/kaggle/working` 스냅샷) → 저장되면 버전 열기 → **Output** 탭에서 `lora_<라운드>.zip` 다운로드  ← 가장 확실
2. 또는 에디터 우측 **Output(`/kaggle/working`) 파일 패널**에서 `lora_<라운드>.zip` 다운로드 (안 보이면 새로고침)
3. 또는 로컬에서 Kaggle API: `kaggle kernels output <user>/<notebook-slug> -p .`

학습 직후 평가(아래 5.5)를 돌렸다면 `logs/eval_<라운드>-qwen.json`의 `time_match_rate` / `final_score`를 셀 출력에서 바로 확인할 수 있다.

In [ ]:
# 참고: FileLink는 Kaggle에서 종종 404. 위 '다운로드 방법'의 Quick Save -> Output 사용 권장.
from IPython.display import FileLink
FileLink(ZIP)